In [0]:
 %pip install phonenumbers pycountry rapidfuzz

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
import unicodedata
import re
import phonenumbers
import pycountry
from rapidfuzz import process

In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_suppliers")
display(df)

In [0]:
#Profiling rapide
null_counts = df.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df.columns
])

display(null_counts)

#doublons exacts
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicates:", total_rows - distinct_rows)
print("Exact duplicates:")

#Doublons exacts
duplicate_rows = df.groupBy(df.columns[1:]).count().filter(F.col("count") > 1).drop("count")
display(duplicate_rows)


In [0]:
#Fonction de normalisation
#trim + suppression accents + remplacement caractères spéciaux + inticap

def normalize_string(value):
    if value is None:
        return None

    value = str(value).strip()
    if value == "":
        return None

    # suppression accents
    value = unicodedata.normalize("NFKD", value).encode("ASCII", "ignore").decode("utf-8")

    # remplacement caractères spéciaux par espace
    value = re.sub(r"[^A-Za-z0-9\s\-']", " ", value)

    # espaces multiples
    value = re.sub(r"\s+", " ", value).strip()

    if value == "":
        return None

    return value.title()

normalize_string_udf = F.udf(normalize_string, StringType())

#Normalisation téléphone

def normalize_phone(phone, country="FR"):
    if phone is None:
        return None

    phone = str(phone).strip()
    if phone == "":
        return None

    try:
        parsed = phonenumbers.parse(phone, country)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(
                parsed,
                phonenumbers.PhoneNumberFormat.E164
            )
    except:
        return None

    return None

normalize_phone_udf = F.udf(normalize_phone, StringType())

#Normalisation country_code
def normalize_country_code(country):
    if country is None:
        return None

    country = str(country).strip()
    if country == "":
        return None

    try:
        return pycountry.countries.lookup(country).alpha_2
    except:
        return None
    
normalize_country_code_udf = F.udf(normalize_country_code, StringType())

In [0]:
#Nettoyage de base
df_clean = (
    df
    .withColumn("supplier_name", normalize_string_udf(F.col("supplier_name")))
    .withColumn("city", normalize_string_udf(F.col("city")))
    .withColumn("country", normalize_string_udf(F.col("country")))
    .withColumn("contact_name", normalize_string_udf(F.col("contact_name")))
    .withColumn("contact_surname", normalize_string_udf(F.col("contact_surname")))
)

In [0]:
# Déduplication métier

# D’abord on parse les dates.

def parse_date_col(col_name):
    c = F.trim(F.col(col_name).cast("string"))

    return F.coalesce(
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-d"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-d"))),

        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/MM/dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/M/d"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/M/dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/MM/d"))),

        F.to_date(F.try_to_timestamp(c, F.lit("dd-MM-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("d-M-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("d-MM-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("dd-M-yyyy"))),

        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-d'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-d'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-dd'T'HH:mm:ss")))
    )

df_clean = (
    df_clean
    .withColumn("created_at", parse_date_col("created_at"))
    .withColumn("updated_at", parse_date_col("updated_at"))
)

# garder la ligne la plus récente selon updated_at.

w = Window.partitionBy("supplier_id").orderBy(F.col("updated_at").desc_nulls_last())

df_clean = (
    df_clean
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
# City : correction par fréquence
# Comme le dataset est petit, on peut récupérer les villes fréquentes côté Python.

city_freq_df = (
    df_clean.groupBy("city")
    .count()
    .filter((F.col("city").isNotNull()) & (F.col("count") > 2))
)

reference_cities = [row["city"] for row in city_freq_df.collect()]
print(reference_cities)

# Fonction de correction
def correct_city(city):
    if city is None:
        return None
    if not reference_cities:
        return city

    match = process.extractOne(city, reference_cities)
    if match and match[1] > 85:
        return match[0]
    return city
    
correct_city_udf = F.udf(correct_city, StringType())
df_clean = df_clean.withColumn("city", correct_city_udf(F.col("city")))

In [0]:
# Country : normalisation + correction fréquence
# Même logique.

country_freq_df = (
    df_clean.groupBy("country")
    .count()
    .filter((F.col("country").isNotNull()) & (F.col("count") > 2))
)

reference_countries = [row["country"] for row in country_freq_df.collect()]
print(reference_countries)
def correct_country(country):
    if country is None:
        return None
    if not reference_countries:
        return country

    match = process.extractOne(country, reference_countries)
    if match and match[1] > 85:
        return match[0]
    return country
correct_country_udf = F.udf(correct_country, StringType())
df_clean = df_clean.withColumn("country", correct_country_udf(F.col("country")))

In [0]:
# Country code
# recalculé à partir de country plutôt que faire confiance à la colonne brute.
df_clean = df_clean.withColumn(
    "country_code",
    normalize_country_code_udf(F.col("country"))
)

In [0]:
#10. Phone number

#Pour que phonenumbers marche mieux, il faut lui donner un pays par défaut cohérent avec le country_code.


def normalize_phone_with_country(phone, country_code):
    if phone is None:
        return None

    phone = str(phone).strip()
    if phone == "":
        return None

    region = country_code if country_code else "FR"

    try:
        parsed = phonenumbers.parse(phone, region)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(
                parsed,
                phonenumbers.PhoneNumberFormat.E164
            )
    except:
        return None

    return None
from pyspark.sql.types import StringType

normalize_phone_with_country_udf = F.udf(normalize_phone_with_country, StringType())
df_clean = df_clean.withColumn(
    "phone_number",
    normalize_phone_with_country_udf(F.col("phone_number"), F.col("country_code"))
)

In [0]:
# Contact email

email_regex = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

df_clean = df_clean.withColumn(
    "contact_email",
    F.when(
        F.lower(F.trim(F.col("contact_email"))).isin("", "null", "n/a", "unknown"),
        None
    ).when(
        F.lower(F.trim(F.col("contact_email"))).rlike(email_regex),
        F.lower(F.trim(F.col("contact_email")))
    ).otherwise(None)
)

In [0]:
df_clean = df_clean.dropDuplicates()

In [0]:
#Contrôles finaux
#Nulls
final_null_counts = df_clean.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df_clean.columns
])

display(final_null_counts)

# supplier_id unique
check_dup = (
    df_clean.groupBy("supplier_id")
    .count()
    .filter(F.col("count") > 1)
)

display(check_dup)
# Écriture Silver
df_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.final_project.silver_suppliers")

In [0]:
became_null = (
    df.alias("before")
    .join(df_clean.alias("after"), on="supplier_id", how="inner")
    .filter(
        F.col("before.contact_name").isNotNull() &
        F.col("after.contact_name").isNull()
    )
    .select(
        "supplier_id",
        F.col("before.contact_name").alias("contact_name_before"),
        F.col("after.contact_name").alias("contact_name_after")
    )
)

display(became_null)

In [0]:
from functools import reduce

cols_to_audit = ["contact_name", "contact_email", "city", "country_code", "phone_number"]

base = df.alias("before").join(df_clean.alias("after"), on="supplier_id", how="inner")

audit_tables = []

for c in cols_to_audit:
    tmp = (
        base
        .filter(
            F.coalesce(F.col(f"before.{c}").cast("string"), F.lit("__NULL__"))
            !=
            F.coalesce(F.col(f"after.{c}").cast("string"), F.lit("__NULL__"))
        )
        .select(
            F.col("supplier_id"),
            F.lit(c).alias("column_name"),
            F.col(f"before.{c}").cast("string").alias("before_value"),
            F.col(f"after.{c}").cast("string").alias("after_value"),
            F.when(
                F.col(f"before.{c}").isNotNull() & F.col(f"after.{c}").isNull(),
                F.lit("TO_NULL")
            ).otherwise(F.lit("CHANGED")).alias("change_type")
        )
    )
    audit_tables.append(tmp)

audit_long = reduce(lambda a, b: a.unionByName(b), audit_tables)

display(audit_long.orderBy("column_name", "supplier_id"))

In [0]:
df_clean.display()